In [32]:
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, length 
from pyspark.sql.types import NumericType, IntegerType, LongType, FloatType, DoubleType, DecimalType
from pyspark.ml.feature import Imputer
import pyspark.sql.functions as F
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [33]:
spark = (
    SparkSession.builder.appName("Cleaning")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9100")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

In [34]:
minio_client = Minio(
    "localhost:9100",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

bucket_name = "pulse-bucket-1"

In [35]:
objects = minio_client.list_objects(bucket_name, prefix="mapped_", recursive=True)
dataframes = {}
for obj in objects:
    df = spark.read.csv(
        f"s3a://{bucket_name}/{obj.object_name}", header=True, inferSchema=True
    )
    object_name = obj.object_name.replace("mapped_", "").replace(".csv", "")
    dataframes[object_name] = df

In [36]:
for table in dataframes.keys():
    df = dataframes[table]

    for column in df.columns:
        if column.endswith("_id"):
            df = df.withColumn(
                column,
                when(
                    regexp_extract(col(column), r"(\d+)", 1) == "",
                    None,
                ).otherwise(regexp_extract(col(column), r"(\d+)", 1)),
            )
            df = df.withColumn(column, col(column))

    dataframes[table] = df
for table, dataframe in dataframes.items():
    print(f"Table: {table}")
    dataframe.show(3)

Table: addresses
+----------+------------------+--------------+-----------+--------------+
|address_id|              city|state_province|postal_code|       country|
+----------+------------------+--------------+-----------+--------------+
|      5555|North Derrickmouth|      Sevilla*|       2944|      Slovénie|
|      5938|             Husum|         Idaho|    PH0 6RX|        Rwanda|
|      5060|              NULL|          NULL|    00000  |United Kingdom|
+----------+------------------+--------------+-----------+--------------+
only showing top 3 rows

Table: categories
+-----------+----------+---------------+
|category_id|  category|   sub_category|
+-----------+----------+---------------+
|        695|  Colthing|      Wholesale|
|        659|Mirrorless|Limited Edition|
|        571|     Suits|      Wholesale|
+-----------+----------+---------------+
only showing top 3 rows

Table: customer_sessions
+----------+-----------+--------------------+--------------------+-----------+-------

cast schema for correct datatypes

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import col

def cast_dataframes(dataframes):
    # 1. Addresses
    if "addresses" in dataframes:
        dataframes["addresses"] = dataframes["addresses"].select(
            col("address_id").cast(StringType()),
            col("city").cast(StringType()),
            col("state_province").cast(StringType()),
            col("postal_code").cast(StringType()),
            col("country").cast(StringType())
        )
        print("Cast addresses DataFrame")

    # 2. Customers
    if "customers" in dataframes:
        dataframes["customers"] = dataframes["customers"].select(
            col("customer_id").cast(StringType()),
            col("gender").cast(StringType()),
            col("date_of_birth").cast(DateType()),
            col("account_status").cast(StringType()),
            col("address_id").cast(StringType()),
            col("city").cast(StringType()),
            col("state_province").cast(StringType()),
            col("postal_code").cast(StringType()),
            col("country").cast(StringType()),
            col("account_created_at").cast(TimestampType()),
            col("last_login_date").cast(DateType()),
            col("is_active").cast(BooleanType())
        )
        print("Cast customers DataFrame")

    # 3. Suppliers
    if "suppliers" in dataframes:
        dataframes["suppliers"] = dataframes["suppliers"].select(
            col("supplier_id").cast(StringType()),
            col("supplier_rating").cast(FloatType()),
            col("supplier_status").cast(StringType()),
            col("is_preferred").cast(BooleanType()),
            col("is_verified").cast(BooleanType()),
            col("contract_start_date").cast(DateType()),
            col("contract_end_date").cast(DateType()),
            col("city").cast(StringType()),
            col("state").cast(StringType()),
            col("zip_code").cast(StringType()),
            col("country").cast(StringType())
        )
        print("Cast suppliers DataFrame")

    # 4. Categories
    if "categories" in dataframes:
        dataframes["categories"] = dataframes["categories"].select(
            col("category_id").cast(StringType()),
            col("category").cast(StringType()),
            col("sub_category").cast(StringType())
        )
        print("Cast categories DataFrame")

    # 5. Products
    if "products" in dataframes:
        dataframes["products"] = dataframes["products"].select(
            col("product_id").cast(StringType()),
            col("product_name").cast(StringType()),
            col("sku").cast(StringType()),
            col("category_id").cast(StringType()),
            col("category").cast(StringType()),
            col("sub_category").cast(StringType()),
            col("brand").cast(StringType()),
            col("supplier_id").cast(StringType()),
            col("cost_price").cast(FloatType()),
            col("sell_price").cast(FloatType()),
            col("launch_date").cast(DateType()),
            col("weight").cast(FloatType()),
            col("dimensions").cast(StringType()),
            col("color").cast(StringType()),
            col("size").cast(StringType()),
            col("material").cast(StringType())
        )
        print("Cast products DataFrame")

    # 6. Inventory
    if "inventory" in dataframes:
        dataframes["inventory"] = dataframes["inventory"].select(
            col("inventory_id").cast(StringType()),
            col("product_id").cast(StringType()),
            col("supplier_id").cast(StringType()),
            col("stock_quantity").cast(IntegerType()),
            col("reserved_quantity").cast(IntegerType()),
            col("minimum_stock_level").cast(IntegerType()),
            col("last_restocked_date").cast(DateType()),
            col("storage_cost").cast(FloatType())
        )
        print("Cast inventory DataFrame")

    # 7. Wishlist
    if "wishlist" in dataframes:
        dataframes["wishlist"] = dataframes["wishlist"].select(
            col("wishlist_id").cast(StringType()),
            col("customer_id").cast(StringType()),
            col("product_id").cast(StringType()),
            col("added_date").cast(DateType()),
            col("purchased_date").cast(DateType()),
            col("removed_date").cast(DateType())
        )
        print("Cast wishlist DataFrame")

    # 8. Shopping Cart
    if "shopping_cart" in dataframes:
        dataframes["shopping_cart"] = dataframes["shopping_cart"].select(
            col("cart_id").cast(StringType()),
            col("customer_id").cast(StringType()),
            col("session_id").cast(StringType()),
            col("product_id").cast(StringType()),
            col("quantity").cast(IntegerType()),
            col("unit_price").cast(FloatType()),
            col("added_date").cast(DateType()),
            col("cart_status").cast(StringType())
        )
        print("Cast shopping_cart DataFrame")

    # 9. Orders
    if "orders" in dataframes:
        dataframes["orders"] = dataframes["orders"].select(
            col("order_id").cast(StringType()),
            col("customer_id").cast(StringType()),
            col("order_status").cast(StringType()),
            col("subtotal").cast(FloatType()),
            col("tax_amount").cast(FloatType()),
            col("shipping_cost").cast(FloatType()),
            col("total_discount").cast(FloatType()),
            col("total_amount").cast(FloatType()),
            col("currency").cast(StringType()),
            col("order_placed_at").cast(TimestampType()),
            col("order_shipped_at").cast(DateType()),
            col("order_delivered_at").cast(DateType())
        )
        print("Cast orders DataFrame")

    # 10. Order Items
    if "order_items" in dataframes:
        dataframes["order_items"] = dataframes["order_items"].select(
            col("order_item_id").cast(StringType()),
            col("order_id").cast(StringType()),
            col("product_id").cast(StringType()),
            col("quantity").cast(IntegerType()),
            col("discount_amount").cast(FloatType()),
            col("product_cost").cast(FloatType())
        )
        print("Cast order_items DataFrame")

    # 11. Payments
    if "payments" in dataframes:
        dataframes["payments"] = dataframes["payments"].select(
            col("payment_id").cast(StringType()),
            col("order_id").cast(StringType()),
            col("payment_method").cast(StringType()),
            col("payment_provider").cast(StringType()),
            col("payment_status").cast(StringType()),
            col("transaction_id").cast(StringType()),
            col("processing_fee").cast(FloatType()),
            col("refund_amount").cast(FloatType()),
            col("refund_date").cast(DateType()),
            col("payment_date").cast(DateType())
        )
        print("Cast payments DataFrame")

    # 12. Reviews
    if "reviews" in dataframes:
        dataframes["reviews"] = dataframes["reviews"].select(
            col("review_id").cast(StringType()),
            col("product_id").cast(StringType()),
            col("customer_id").cast(StringType()),
            col("rating").cast(IntegerType()),
            col("review_title").cast(StringType()),
            col("review_desc").cast(StringType()),
            col("review_date").cast(TimestampType())
        )
        print("Cast reviews DataFrame")

    # 13. Marketing Campaigns
    if "marketing_campaigns" in dataframes:
        dataframes["marketing_campaigns"] = dataframes["marketing_campaigns"].select(
            col("campaign_id").cast(StringType()),
            col("campaign_name").cast(StringType()),
            col("campaign_type").cast(StringType()),
            col("start_date").cast(DateType()),
            col("end_date").cast(DateType()),
            col("budget").cast(FloatType()),
            col("spent_amount").cast(FloatType()),
            col("impressions").cast(IntegerType()),
            col("clicks").cast(IntegerType()),
            col("conversions").cast(IntegerType()),
            col("target_audience").cast(StringType()),
            col("campaign_status").cast(StringType())
        )
        print("Cast marketing_campaigns DataFrame")

    # 14. Customer Sessions
    if "customer_sessions" in dataframes:
        dataframes["customer_sessions"] = dataframes["customer_sessions"].select(
            col("session_id").cast(StringType()),
            col("customer_id").cast(StringType()),
            col("session_start").cast(TimestampType()),
            col("session_end").cast(TimestampType()),
            col("device_type").cast(StringType()),
            col("referrer_source").cast(StringType()),
            col("pages_viewed").cast(IntegerType()),
            col("products_viewed").cast(IntegerType()),
            col("conversion_flag").cast(BooleanType()),
            col("cart_abandonment_flag").cast(BooleanType())
        )
        print("Cast customer_sessions DataFrame")

    print("\n✅ All DataFrames cast successfully!")
    return dataframes

dataframes = cast_dataframes(dataframes)

Cast addresses DataFrame
Cast customers DataFrame
Cast suppliers DataFrame
Cast categories DataFrame
Cast products DataFrame
Cast inventory DataFrame
Cast wishlist DataFrame
Cast shopping_cart DataFrame
Cast orders DataFrame
Cast order_items DataFrame
Cast payments DataFrame
Cast reviews DataFrame
Cast marketing_campaigns DataFrame
Cast customer_sessions DataFrame

✅ All DataFrames cast successfully!


In [38]:
dataframes["customers"].columns

['customer_id',
 'gender',
 'date_of_birth',
 'account_status',
 'address_id',
 'city',
 'state_province',
 'postal_code',
 'country',
 'account_created_at',
 'last_login_date',
 'is_active']

Merging  addresses table with Customers
and categories table with products if exists

In [39]:
def merge():
      if not "addresses" in dataframes:
            print("Addresses DataFrame is missing.")

      if "addresses" in dataframes and "customers" in dataframes:
            dataframes["addresses"].createOrReplaceTempView("addresses")
            dataframes["customers"].createOrReplaceTempView("customers")

            customers = spark.sql("""
            SELECT c.customer_id, c.gender, c.date_of_birth, c.account_status,
                  a.city, a.state_province, a.postal_code, a.country,
                  c.account_created_at,c.last_login_date,c.is_active
            FROM customers c
            LEFT JOIN addresses a
                  ON c.address_id = a.address_id
            """)
            dataframes["customers"] = customers
            print("Merged addresses into customers.")
            dataframes.pop("addresses", None)

      if not "categories" in dataframes:
            print("Categories DataFrame is missing.")
            return
      
      if "categories" in dataframes and "products" in dataframes:
            dataframes["categories"].createOrReplaceTempView("categories")
            dataframes["products"].createOrReplaceTempView("products")

            products = spark.sql("""
            SELECT p.product_id, p.product_name, p.sku, cat.category, cat.sub_category,
                  p.brand, p.supplier_id, p.cost_price, p.sell_price, p.launch_date,
                  p.weight, p.dimensions, p.color, p.size, p.material
            FROM products p
            LEFT JOIN categories cat
                  ON p.category_id = cat.category_id
            """)
            dataframes["products"] = products
            print("Merged categories into products.")
            dataframes.pop("categories", None)
merge()

Merged addresses into customers.
Merged categories into products.


# Cleaning null values and Duplicate values 

In [40]:
def check_dups():
    for name, df in dataframes.items():
        dup_rows = df.groupBy(*df.columns).count().filter("count > 1")
        row_count = dup_rows.count()
        print(f"The number of duplicate rows in {name} is: {row_count}")
check_dups()

The number of duplicate rows in customer_sessions is: 72
The number of duplicate rows in customers is: 28
The number of duplicate rows in inventory is: 27
The number of duplicate rows in marketing_campaigns is: 10
The number of duplicate rows in order_items is: 101
The number of duplicate rows in orders is: 28
The number of duplicate rows in payments is: 46
The number of duplicate rows in products is: 18
The number of duplicate rows in reviews is: 40
The number of duplicate rows in shopping_cart is: 47
The number of duplicate rows in suppliers is: 14
The number of duplicate rows in wishlist is: 35


In [41]:
def drop_dups():
    for table in dataframes.keys():
        dataframes[table] = dataframes[table].dropDuplicates()
drop_dups()


In [42]:
check_dups()

The number of duplicate rows in customer_sessions is: 0
The number of duplicate rows in customers is: 0
The number of duplicate rows in inventory is: 0
The number of duplicate rows in marketing_campaigns is: 0
The number of duplicate rows in order_items is: 0
The number of duplicate rows in orders is: 0
The number of duplicate rows in payments is: 0
The number of duplicate rows in products is: 0
The number of duplicate rows in reviews is: 0
The number of duplicate rows in shopping_cart is: 0
The number of duplicate rows in suppliers is: 0
The number of duplicate rows in wishlist is: 0


In [43]:
def drop_null_rows(table, col_name):
    if table in dataframes:
        df = dataframes[table]
        if col_name in df.columns:
            before = df.count()
            cleaned = df.filter(F.col(col_name).isNotNull())
            dataframes[table] = cleaned
            after = cleaned.count()
            print(f"Removed {before - after} rows from '{table}' where '{col_name}' is NULL")
        else:
            print(f"Column '{col_name}' not found in '{table}'")    
    else:
        print(f"Table '{table}' not found in dataframes")

Dropping all rows from all tables where primary key and foreign keys are null 

In [44]:
all_ids = ["session_id","customer_id", "address_id", "product_id", "supplier_id", "order_id", "order_item_id", "payment_id", "campaign_id","cart_id", "review_id", "wishlist_id"]
for table in dataframes.keys():
    for col in dataframes[table].columns:
        if col in all_ids:
            drop_null_rows(table, col)

Removed 51 rows from 'customer_sessions' where 'session_id' is NULL
Removed 1854 rows from 'customer_sessions' where 'customer_id' is NULL
Removed 10 rows from 'customers' where 'customer_id' is NULL
Removed 41 rows from 'inventory' where 'product_id' is NULL
Removed 70 rows from 'inventory' where 'supplier_id' is NULL
Removed 20 rows from 'marketing_campaigns' where 'campaign_id' is NULL
Removed 38 rows from 'order_items' where 'order_item_id' is NULL
Removed 127 rows from 'order_items' where 'order_id' is NULL
Removed 104 rows from 'order_items' where 'product_id' is NULL
Removed 30 rows from 'orders' where 'order_id' is NULL
Removed 35 rows from 'orders' where 'customer_id' is NULL
Removed 42 rows from 'payments' where 'payment_id' is NULL
Removed 81 rows from 'payments' where 'order_id' is NULL
Removed 14 rows from 'products' where 'product_id' is NULL
Removed 41 rows from 'products' where 'supplier_id' is NULL
Removed 29 rows from 'reviews' where 'review_id' is NULL
Removed 71 row

In [45]:
def check_nulls():
    for df in dataframes.values():
        null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
        null_counts.show()
check_nulls()

+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+
|session_id|customer_id|session_start|session_end|device_type|referrer_source|pages_viewed|products_viewed|conversion_flag|cart_abandonment_flag|
+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+
|         0|          0|          263|        118|        120|             87|         171|             97|            128|                   25|
+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+

+-----------+------+-------------+--------------+----+--------------+-----------+-------+------------------+---------------+---------+
|customer_id|gender|date_of_birth|account_status|city|state_province|postal_code|country|account_created_at|last_login_date|is_active|

Filling null values fo non numeric columns 

In [46]:
def fill_null():
    if "customers" in dataframes.keys():
        dataframes["customers"] = dataframes["customers"].fillna({
            "gender": "Unknown",
            "account_status": "Unknown",
            "city": "Unknown",
            "state_province": "Unknown",
            "postal_code": "00000",
            "country": "Unknown",
            "date_of_birth": "1900-01-01",
            "account_created_at": "1900-01-01",
            "last_login_date": "1900-01-01",
            "is_active": "false"
        })
    else:
        print("Customers DataFrame is missing.")

    if "suppliers" in dataframes.keys():
        dataframes["suppliers"] = dataframes["suppliers"].fillna({
            "supplier_rating": 0.0,
            "supplier_status": "Unknown",
            "is_preferred": "false",
            "is_verified": "false",
            "contract_start_date": "1900-01-01",
            "contract_end_date": "1900-01-01",
            "city": "Unknown",
            "state": "Unknown",
            "zip_code": "00000",
            "country": "Unknown",
        })
    else:
        print("Suppliers DataFrame is missing.")

    if "products" in dataframes.keys():
        dataframes["products"] = dataframes["products"].fillna({
            "product_name": "Unknown",
            "sku": "Unknown",
            "category": "Unknown",
            "sub_category": "Unknown",
            "brand": "Unknown",
            "launch_date": "1900-01-01",
            "weight": "0.0",
            "dimensions": "Unknown",
            "color": "Unknown",
            "size": "Unknown",
            "material": "Unknown"
        })
    else:
        print("Products DataFrame is missing.")

    if "wishlist" in dataframes.keys():
        dataframes["wishlist"] = dataframes["wishlist"].fillna({
            "added_date": "1900-01-01",
            "purchased_date": "1900-01-01",
            "removed_date": "1900-01-01"
        })
    else:
        print("Wishlist DataFrame is missing.")

    if "shopping_cart" in dataframes.keys():
        dataframes["shopping_cart"] = dataframes["shopping_cart"].fillna({
            "added_date": "1900-01-01",
            "cart_status": "Unknown"
        })
    else:
        print("Shopping Cart DataFrame is missing.")    
        
    if "inventory" in dataframes.keys():
        dataframes["inventory"] = dataframes["inventory"].fillna({
            "last_restocked_date": "1900-01-01"
        })
    else:
        print("Inventory DataFrame is missing.")

    if "customer_sessions" in dataframes.keys():
        dataframes["customer_sessions"] = dataframes["customer_sessions"].fillna({
            "session_start": "1900-01-01",
            "session_end": "1900-01-01",
            "device_type": "Unknown",
            "referrer_source": "Unknown",
            "pages_viewed": 0,
            "products_viewed": 0,
            "conversion_flag": "false",
            "cart_abandonment_flag": "false"
        })
    else:
        print("Customer Sessions DataFrame is missing.")

    if "reviews" in dataframes.keys():
        dataframes["reviews"] = dataframes["reviews"].fillna({
            "review_date": "1900-01-01",
            "review_title": "Unknown",
            "review_desc": "Unknown"
        })
    else:
        print("Reviews DataFrame is missing.")

    if "orders" in dataframes.keys():
        dataframes["orders"] = dataframes["orders"].fillna({
            "order_status": "Unknown",
            "order_placed_at": "1900-01-01",
            "order_shipped_at": "1900-01-01",
            "order_delivered_at": "1900-01-01",
            "currency": "Unknown",
        })
    else:
        print("Orders DataFrame is missing.")

    if "payments" in dataframes.keys():
        dataframes["payments"] = dataframes["payments"].fillna({
            "payment_method": "Unknown",
            "payment_status": "Unknown",
            "payment_date": "1900-01-01",
            "transaction_id": "Unknown ",
            "payment_provider": "Unknown",
            "refund_date": "1900-01-01"
        })
    else:
        print("Payments DataFrame is missing.")

    if "marketing_campaigns" in dataframes.keys():
        dataframes["marketing_campaigns"] = dataframes["marketing_campaigns"].fillna({
            "start_date": "1900-01-01",
            "end_date": "1900-01-01",
            "campaign_type": "Unknown",
            "campaign_status": "Unknown",
            "campaign_name": "Unknown",
            "target_audience": "Unknown",
            "impressions": 0,
            "clicks": 0,
            "conversions": 0
        })
    else:
        print("Marketing Campaigns DataFrame is missing.")

fill_null()

Imputing missing numeric values

In [47]:


def impute_missing_values(table, numeric_cols):

    total_rows = dataframes[table].count()
    print(f"Total rows: {total_rows}")

    non_null_counts = dataframes[table].select([F.count(F.col(c)).alias(c) for c in numeric_cols]).collect()[0]

    all_null_cols = []
    valid_numeric_cols = []

    for col_name in numeric_cols:
        non_null_count = non_null_counts[col_name]
        if non_null_count == 0:
            all_null_cols.append(col_name)
            print(f"🚫 {col_name}: ALL NULL - will fill with 0")
        else:
            valid_numeric_cols.append(col_name)
            null_count = total_rows - non_null_count
            print(f"✅ {col_name}: {non_null_count} non-null, {null_count} null - will impute")

    print(f"\nAll-NULL columns: {all_null_cols}")
    print(f"Valid columns for imputation: {valid_numeric_cols}")


    if all_null_cols:
        fill_dict = {col: 0 for col in all_null_cols}
        dataframes[table] = dataframes[table].fillna(fill_dict)
        print(f"✅ Filled all-NULL columns with 0: {all_null_cols}")


    if valid_numeric_cols:
        imputer = Imputer(inputCols=valid_numeric_cols, outputCols=valid_numeric_cols).setStrategy("median")
        dataframes[table] = imputer.fit(dataframes[table]).transform(dataframes[table])
        print(f"✅ Successfully imputed columns with median: {valid_numeric_cols}")
    else:
        print("⚠️ No valid columns found for imputation")

    print("\n" + "="*50)
    print("🔍 Final check for NULL values in inventory:")
    print("="*50)


In [48]:
def impute_all():
    all_ids = ["session_id","customer_id", "address_id", "product_id", "supplier_id", "order_id", "order_item_id", "payment_id", "campaign_id","cart_id", "review_id", "wishlist_id"]
    for table in dataframes.keys():
        numeric_cols = [field.name for field in dataframes[table].schema.fields 
                        if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType, DecimalType))] 
        numeric_cols = [col for col in numeric_cols if col not in all_ids]
        if numeric_cols:
            print(f"\nImputing missing values for table: {table}")
            impute_missing_values(table, numeric_cols)
        else:
            print(f"\nNo numeric columns found in table: {table}, skipping imputation.")

impute_all()


Imputing missing values for table: customer_sessions
Total rows: 3135
✅ pages_viewed: 3135 non-null, 0 null - will impute
✅ products_viewed: 3135 non-null, 0 null - will impute

All-NULL columns: []
Valid columns for imputation: ['pages_viewed', 'products_viewed']
✅ Successfully imputed columns with median: ['pages_viewed', 'products_viewed']

🔍 Final check for NULL values in inventory:

No numeric columns found in table: customers, skipping imputation.

Imputing missing values for table: inventory
Total rows: 1390
✅ stock_quantity: 1363 non-null, 27 null - will impute
🚫 reserved_quantity: ALL NULL - will fill with 0
✅ minimum_stock_level: 1363 non-null, 27 null - will impute
✅ storage_cost: 1362 non-null, 28 null - will impute

All-NULL columns: ['reserved_quantity']
Valid columns for imputation: ['stock_quantity', 'minimum_stock_level', 'storage_cost']
✅ Filled all-NULL columns with 0: ['reserved_quantity']
✅ Successfully imputed columns with median: ['stock_quantity', 'minimum_stoc

In [49]:
check_nulls()

+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+
|session_id|customer_id|session_start|session_end|device_type|referrer_source|pages_viewed|products_viewed|conversion_flag|cart_abandonment_flag|
+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+
|         0|          0|            0|          0|          0|              0|           0|              0|              0|                    0|
+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+

+-----------+------+-------------+--------------+----+--------------+-----------+-------+------------------+---------------+---------+
|customer_id|gender|date_of_birth|account_status|city|state_province|postal_code|country|account_created_at|last_login_date|is_active|

Standardizing Numerical columns by removing outliers 

In [50]:
def remove_outliers(table_name, columns):
    if table_name not in dataframes:
        print(f"Table {table_name} not found")
        return
    
    df = dataframes[table_name]
    result_df = df
    
    for column in columns:
        if column not in df.columns:
            print(f"Column {column} not found in {table_name}")
            continue
            
        print(f"\nProcessing outliers for {column} in {table_name}...")
        quantiles = result_df.approxQuantile(column, [0.01, 0.99], 0.0)
        low_cutoff, high_cutoff = quantiles[0], quantiles[1]
        
        print(f"  {column} - Low cutoff: {low_cutoff}, High cutoff: {high_cutoff}")
        if low_cutoff < 0:
            low_cutoff = 0
            print(f"  Adjusted Low cutoff for {column} to 0 since it was negative.")
            
        before_count = result_df.count()
        result_df = result_df.filter(
            (F.col(column) >= low_cutoff) & (F.col(column) <= high_cutoff)
        )
        after_count = result_df.count()
        
        removed = before_count - after_count
        print(f"  Removed {removed} outlier rows based on {column}")
        
        
       
    dataframes[table_name] = result_df
    print(f"\n✅ Completed outlier removal for {table_name}")
   


In [51]:
def remove_all_outliers():
    all_ids = ["session_id","customer_id", "address_id", "product_id", "supplier_id", "order_id", "order_item_id", "payment_id", "campaign_id","cart_id", "review_id", "wishlist_id"]
    for table in dataframes.keys():
        numeric_cols = [field.name for field in dataframes[table].schema.fields
                        if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType, DecimalType))
                       ]
        numeric_cols = [column_name for column_name in numeric_cols if column_name not in all_ids]
        if numeric_cols:
            print(f"\nRemoving outliers for table: {table}")  
            remove_outliers(table, numeric_cols)
        else:
            print(f"\nNo numeric columns found in table: {table}, skipping outlier removal.")

remove_all_outliers()


Removing outliers for table: customer_sessions

Processing outliers for pages_viewed in customer_sessions...
  pages_viewed - Low cutoff: -4.0, High cutoff: 30.0
  Adjusted Low cutoff for pages_viewed to 0 since it was negative.
  Removed 71 outlier rows based on pages_viewed

Processing outliers for products_viewed in customer_sessions...
  products_viewed - Low cutoff: 0.0, High cutoff: 15.0
  Removed 30 outlier rows based on products_viewed

✅ Completed outlier removal for customer_sessions

No numeric columns found in table: customers, skipping outlier removal.

Removing outliers for table: inventory

Processing outliers for stock_quantity in inventory...
  stock_quantity - Low cutoff: 0.0, High cutoff: 952.0
  Removed 13 outlier rows based on stock_quantity

Processing outliers for reserved_quantity in inventory...
  reserved_quantity - Low cutoff: 0.0, High cutoff: 0.0
  Removed 0 outlier rows based on reserved_quantity

Processing outliers for minimum_stock_level in inventory..

In [ ]:
def validate_dates_and_timestamps():
    
    from pyspark.sql.functions import current_date, current_timestamp, when, col
    from pyspark.sql.types import DateType, TimestampType
    
    print("🕒 Validating dates and timestamps...")
    
    for table_name, df in dataframes.items():
        print(f"\n📅 Processing {table_name}...")
        
        
        date_timestamp_cols = []
        for field in df.schema.fields:
            if isinstance(field.dataType, (DateType, TimestampType)):
                date_timestamp_cols.append((field.name, field.dataType))
        
        if not date_timestamp_cols:
            print(f"  ✅ No date/timestamp columns found in {table_name}")
            continue
            
        result_df = df
        
        for col_name, col_type in date_timestamp_cols:
            print(f"  🔍 Checking {col_name} ({col_type})...")
            
            if isinstance(col_type, DateType):
                
                future_count = result_df.filter(col(col_name) > current_date()).count()
                
                if future_count > 0:
                    print(f"    ⚠️ Found {future_count} future dates in {col_name}")
                    result_df = result_df.withColumn(
                        col_name,
                        when(col(col_name) > current_date(), current_date())
                        .otherwise(col(col_name))
                    )
                    print(f"    ✅ Updated {future_count} future dates to current date")
                else:
                    print(f"    ✅ No future dates found in {col_name}")
                    
            elif isinstance(col_type, TimestampType):
                
                future_count = result_df.filter(col(col_name) > current_timestamp()).count()
                
                if future_count > 0:
                    print(f"    ⚠️ Found {future_count} future timestamps in {col_name}")
                    result_df = result_df.withColumn(
                        col_name,
                        when(col(col_name) > current_timestamp(), current_timestamp())
                        .otherwise(col(col_name))
                    )
                    print(f"    ✅ Updated {future_count} future timestamps to current timestamp")
                else:
                    print(f"    ✅ No future timestamps found in {col_name}")
        
        dataframes[table_name] = result_df
    
    print("\n🎉 Date and timestamp validation completed!")


validate_dates_and_timestamps()

🕒 Validating dates and timestamps...

📅 Processing customer_sessions...
  🔍 Checking session_start (TimestampType())...
    ⚠️ Found 3 future timestamps in session_start
    ✅ Updated 3 future timestamps to current timestamp
  🔍 Checking session_end (TimestampType())...
    ⚠️ Found 3 future timestamps in session_end
    ✅ Updated 3 future timestamps to current timestamp

📅 Processing customers...
  🔍 Checking date_of_birth (DateType())...
    ✅ No future dates found in date_of_birth
  🔍 Checking account_created_at (TimestampType())...
    ✅ No future timestamps found in account_created_at
  🔍 Checking last_login_date (TimestampType())...
    ⚠️ Found 7 future timestamps in last_login_date
    ✅ Updated 7 future timestamps to current timestamp

📅 Processing inventory...
  🔍 Checking last_restocked_date (TimestampType())...
    ✅ No future timestamps found in last_restocked_date

📅 Processing marketing_campaigns...
  🔍 Checking start_date (DateType())...
    ⚠️ Found 2 future dates in s